In [30]:
import pandas as pd
import altair as alt
import json
import vl_convert as vlc
import time
from PIL import Image


df=pd.DataFrame({'Org': ['Social 1&2','Social 1&2','Social 1&2','Social 1&2', 'Social 1&2','Recall 1&2', 'Recall 1&2', 'Recall 1&2', 'Recall 1&2', 'Recall 1&2','Recall 3&4', 'Recall 3&4', 'Recall 3&4', 'Recall 3&4','Recall 3&4'],
                 'Response profile': ['CS-', 'CS+', 'Mixed', 'Non-soc', 'Non','CS-', 'CS+', 'Mixed', 'Non-soc','Non', 'CS-', 'CS+', 'Mixed', 'Non-soc','Non'],
                'numbers': [10,7,8,21,54,9,8,11,20,52,7,8,10,22,53]})


# Change bar size
bar_size = 50

## colors
# purple #6e05ab
# light blue #2462bf
# lightgrey
# blue-purple: #3f3999
value_map = {
    'CS+' : ['CS+ selective','#22690d'],
    'Non' : ['Non-responsive','lightgrey'],
    'Non-soc': ['Non-social','white'],
    'Mixed' : ['Mixed selectivity','#518a88'],
    'CS-' : ['CS- selective','#4bc227'],
}

df['Response profile'] = df['Response profile'].apply(lambda x: value_map[x][0])
domain = [i[0] for i in value_map.values()]
range = [i[1] for i in value_map.values()]



# Set font sizes
axis_labelFontSize = 22
title_FontSize = 22
text_FontSize = 22
legend_labelFontSize = 22
legend_titleFontSize = 22



# Or you can specify the order manually:
sort_order = ['Social 1&2','Recall 1&2','Recall 3&4']



### Direction of bars
direction = 'horizontal'  # or 'vertical'
#direction = 'vertical'  # or 'vertical'

if direction == 'horizontal':
    ascending=[True, True]

    x_bar_str = 'sum(numbers)'
    x_text_str = 'position'
    y_bar_str = 'Org'
    y_text_str = 'Org'
    width=400
    height=240
    x_title = ''
    y_title = None
else:
    ascending=[True, False]
    x_bar_str = 'Org'
    y_bar_str = 'sum(numbers)'
    x_text_str = 'Org'
    y_text_str = 'position'
    width=240
    height=360
    x_title = None
    y_title = '[]'

# Need to calculate correct postion to show text (middle of bars)
df.sort_values(by=['Org','Response profile'],ascending=ascending,inplace=True)
df['cul'] = df.groupby(['Org'])['numbers'].cumsum()
df['position'] = df.apply(lambda row: row['cul'] - row['numbers'] // 2,axis=1)

### Plot
bars=alt.Chart().mark_bar(stroke='black', strokeWidth=4, opacity=1, fillOpacity = 0.3, size=bar_size).encode(
    x=alt.X(x_bar_str, title=''),
    y=alt.Y(y_bar_str, title=None, sort=sort_order),
    color=alt.Color('Response profile', title=None,
                    scale=alt.Scale(domain=domain,
                                    range=range),
                    legend=alt.Legend(title='Response profile', orient = 'right',labelFontSize=legend_labelFontSize,titleFontSize=legend_titleFontSize))
)

text=alt.Chart().mark_text(align='center', baseline='middle', size=text_FontSize).encode(
    y=alt.Y(y_text_str, title=None, sort=sort_order),
    x=x_text_str,
    text='numbers')

chart_out = alt.layer(bars, text, data=df).properties(
    width=360,
    height=240
    #width=240,
   # height=360
).configure_axis(
    labelFontSize = axis_labelFontSize,
    titleFontSize = title_FontSize,
).configure_legend(labelLimit= 0,titleLimit=0)

chart_out

alt.LayerChart(...)

In [31]:
# Save as jpg and eps
# No direct way to save an altair chart to desired format.
# json -> png -> jpg & eps
chart_out.save('temp_json.json')
with open('temp_json.json') as f:
    temp_json = json.load(f)

time.sleep(5)
png_data = vlc.vegalite_to_png(vl_spec=temp_json, scale=2)
with open('chart.png', "wb") as f:
    f.write(png_data)

im = Image.open(r'chart.png')
rgb_im = im.convert('RGB')
rgb_im.save(r'BarplotMockSFC.jpg')
rgb_im.save('BarplotMockSFC.eps')